# Hosting your agent

You've built a research agent in [notebook 00](./00_The_one_liner_research_agent.ipynb). It runs on your laptop. Now someone else needs to use it — a teammate, a cron job, a web app, a customer. That means it has to run *somewhere other than your terminal*, stay up, keep conversations alive across restarts, and not leak your API key.

This notebook takes the exact same agent and deploys it through three tiers:

| Tier | Where it runs | When to use it |
|---|---|---|
| **1. Docker** | Your machine / a single VM | Dev loop, internal tools, single-tenant |
| **2. Modal** | Managed serverless | You want a URL and scale-to-zero without managing infra |
| **3. Kubernetes** | Your own cluster | Multi-tenant, regulated environments, full control |

The agent code, the container image, and the HTTP interface are **identical** across all three. Only the operational machinery around the container changes. That's the point: once you've containerized the agent and given it a stable interface, choosing a host is a deployment decision, not a rewrite.

> **Cost to run this notebook end-to-end:** roughly **$TBD** in Anthropic API calls plus **$TBD** in Modal compute. Every tier has a teardown step.

All the deployment code lives in [`hosting/`](./hosting/) next to this notebook.

## Before you start: should you be using the Agent SDK?

If you're building a **customer-facing chat product**, look at [Claude Managed Agents](https://platform.claude.com/docs/en/agents) first — you get hosting, sessions, and a UI out of the box and skip most of this notebook.

The Agent SDK is the right choice when you need **programmatic control**: batch and job-shaped agents, internal tools, agents embedded in your own backend, or regulated environments where you have to own the infrastructure. If that's you, read on.

## The mental model

Three nouns to keep straight:

- A **process** is one running Python interpreter with the SDK loaded. It talks to the Anthropic API.
- A **session** is one conversation — the prompt history, tool calls, and results that the SDK writes to disk so you can `resume=` it later.
- A **container** is a packaged process plus its filesystem: your agent code, the SDK, Node, and a place for sessions to live.

Unlike in-process SDKs (OpenAI Agents SDK, Google ADK) where an "agent" is an object you instantiate inside your web server, a Claude Agent SDK agent **is** a process. That makes isolation trivial (one container = one blast radius) but means hosting is a distributed-systems problem, not a `pip install` problem.

Every deployment, at any tier, has to do the same four jobs:

```
┌────────────────────────────────────────────────────────────────────┐
│  caller ──► gateway ──► [ spawn | route ] ──► agent container ──► API
│                 │                                     │
│                 └── auth (not the agent's job)        └── /data  (persist)
│                                                                      │
│  orchestrator ──────────────── lifecycle ────────────────────────────┘
└────────────────────────────────────────────────────────────────────┘
```

1. **Spawn** a container when work arrives
2. **Route** each request to the container that owns that session
3. **Lifecycle** — kill idle containers, restart crashed ones
4. **Persist** session transcripts so a restart doesn't lose the conversation

Tier 1 does all four by hand. Tier 2 delegates spawn+lifecycle to Modal. Tier 3 delegates all four to Kubernetes plus a small gateway. The agent container never changes.

## The agent we're deploying

We're reusing [`research_agent/agent.py`](./research_agent/agent.py) from notebook 00 — the one-liner research agent with `WebSearch` and `Read`. If you haven't done notebook 00, do it first; this notebook assumes the agent already works.

The only thing `hosting/` adds is a thin HTTP server and a Dockerfile. The agent's options come straight from `research_agent.agent` — we import them, we don't redefine them:

In [ ]:
from research_agent.agent import RESEARCH_SYSTEM_PROMPT, DEFAULT_MODEL
print(f"model: {DEFAULT_MODEL}")
print(RESEARCH_SYSTEM_PROMPT)

### Setup

Create `hosting/.env` with your API key. This file is gitignored.

In [ ]:
%%bash
test -f hosting/.env || cp hosting/.env.example hosting/.env
echo 'Edit hosting/.env and set ANTHROPIC_API_KEY, then re-run this cell.'
grep -q '^ANTHROPIC_API_KEY=sk-ant-' hosting/.env \
  && ! grep -q 'your-key-here' hosting/.env \
  && echo '✅ key looks set'

---
## Tier 1a — Ephemeral: one prompt, one container, done

The simplest possible deployment: a container that runs the agent **once** on a prompt from an env var, prints the result, and exits. No server, no sessions, no state.


> **Model note:** the hosting layer defaults to `claude-sonnet-4-6` so your test requests stay cheap while you work through this notebook. Set `MODEL=claude-opus-4-6` in `hosting/.env` to match notebook 00's config exactly.
This is genuinely enough for a lot of real work — invoice processing, nightly report generation, batch translation, one-off analysis. If your agent's job is "take input, produce output, stop," you don't need anything past this section.

The [`Dockerfile`](./hosting/Dockerfile) packages the agent, the SDK, and the Claude Code CLI the SDK drives under the hood. The build context is `claude_agent_sdk/` (this directory), not `hosting/`, because the image needs `research_agent/` and `utils/` too:

In [ ]:
%%bash
docker build -f hosting/Dockerfile -t research-agent . | tail -n 3

In [ ]:
%%bash
docker run --rm --env-file hosting/.env \
  -e PROMPT='What is the Claude Agent SDK, in one paragraph?' \
  research-agent

That's it. `entrypoint.sh` sees no `serve` argument, so [`run_once.py`](./hosting/run_once.py) calls `research_agent.agent.send_query()` with `$PROMPT` and exits 0.

**When this is enough:** job-shaped tasks where each invocation is independent. Run it from a cron, a queue worker, a CI step — anywhere you'd run any other CLI.

---
## Tier 1b — Hybrid: add a server so conversations can continue

Ephemeral mode can't hold a conversation — every `docker run` is a fresh session. For a chat-shaped agent you need a long-lived process that accepts follow-ups and resumes the right session each time.

[`hosting/server.py`](./hosting/server.py) is a ~100-line FastAPI app that does exactly that and nothing more. The interface is the contract every tier conforms to:

```
GET  /health                            → 200 {"status": "ok"}
POST /sessions/{session_id}/messages    → 200 text/event-stream of SDK messages
```

Two things worth noticing in `server.py`:

- **The server has no auth.** The docstring says so loudly. Auth is the gateway's job (tier 3 shows where it goes). The server validates `session_id` format and trusts the caller.
- **It keeps a small map from your `session_id` to the SDK's internal one.** The SDK generates its own session IDs; you can't choose them. The server learns the SDK's ID from the first turn's `ResultMessage` and passes it to `resume=` on follow-ups. The map is persisted next to the transcripts under `/data`.

Start it with docker-compose, which also mounts `./sessions` at `/data` so transcripts survive restarts:

In [ ]:
%%bash
cd hosting/docker && docker compose up --build -d
sleep 3
curl -s http://localhost:8000/health

Send a prompt and stream the response (`-N` disables curl's buffering so you see events as they arrive):

In [ ]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"What are the three most interesting AI agent trends right now?"}'

Now a follow-up to the **same** `session_id`. The agent remembers the first turn — that's `resume=` working:

In [ ]:
%%bash
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Tell me more about the second one."}'

Restart the container and send *another* follow-up. The volume mount kept `/data`, so the conversation survives:

In [ ]:
%%bash
cd hosting/docker && docker compose restart && sleep 3
curl -N -s -X POST http://localhost:8000/sessions/demo-1/messages \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Summarize what we have discussed so far."}'

In [ ]:
%%bash
# Teardown tier 1
cd hosting/docker && docker compose down

---
## Tier 2 — Modal: same image, now it's a URL

Tier 1 runs on your machine. Tier 2 runs the **same Dockerfile** on [Modal](https://modal.com) via `modal.Sandbox`, which gives you a public HTTPS URL, scale-to-zero, and no servers to manage.

That URL is *public* — anyone who has it can spend your API budget. Tiers 1 and 3 assume an authenticating gateway in front; tier 2 has no gateway, so `modal_app.py` generates a per-deploy bearer token and passes it as `AGENT_AUTH_TOKEN`. `server.py` only enforces the token when that env var is set, so the other tiers are unaffected.

[`hosting/modal/modal_app.py`](./hosting/modal/modal_app.py) is short because nothing about the agent changes:

```python
auth_token = secrets.token_urlsafe(32)
image = modal.Image.from_dockerfile("hosting/Dockerfile", context_dir=".")
sandbox = modal.Sandbox.create(
    "./hosting/entrypoint.sh", "serve",
    image=image,
    secrets=[
        modal.Secret.from_name("anthropic"),
        modal.Secret.from_dict({"AGENT_AUTH_TOKEN": auth_token}),
    ],
    volumes={"/data": sessions_volume},
    encrypted_ports=[8000],
)
print(sandbox.tunnels()[8000].url)
```

Persistence uses a `modal.Volume` mounted at `/data` — the same `CLAUDE_CONFIG_DIR` trick as tier 1. (If your workload has many sandboxes writing concurrently and you hit Volume commit-semantics issues, swap in a [`SessionStore`](https://code.claude.com/docs/en/agent-sdk/session-storage); that's also what tier 3 and production deployments use.)

One-time setup:

One-time setup, **in your terminal** (`modal setup` opens a browser, so it can't run from a notebook cell):

```bash
pip install modal
modal setup
```

Then create the secret Modal will inject as `ANTHROPIC_API_KEY`:

In [ ]:
%%bash
modal secret create anthropic ANTHROPIC_API_KEY="$(grep ANTHROPIC_API_KEY hosting/.env | cut -d= -f2)"

In [ ]:
%%bash
python hosting/modal/modal_app.py | tee /tmp/modal_deploy.out
MODAL_URL=$(awk '/^url:/ {print $2}' /tmp/modal_deploy.out)
MODAL_TOKEN=$(awk '/^token:/ {print $2}' /tmp/modal_deploy.out)
{ echo "MODAL_URL=$MODAL_URL"; echo "MODAL_TOKEN=$MODAL_TOKEN"; } > /tmp/modal_url.env

In [ ]:
%%bash
source /tmp/modal_url.env
curl -N -s -X POST "$MODAL_URL/sessions/demo-1/messages" \
  -H "Authorization: Bearer $MODAL_TOKEN" \
  -H 'Content-Type: application/json' \
  -d '{"prompt":"Give me a one-sentence summary of the Claude Agent SDK."}'

Same interface, same image, different host. When nothing's calling it, Modal scales the sandbox to zero and you pay nothing.

Teardown so you aren't billed for idle resources:

In [ ]:
%%bash
python hosting/modal/teardown.py

---
## Tier 3 — Kubernetes: when you need to own the infrastructure

Tier 3 is for multi-tenant production, regulated environments, or anywhere you need full control over networking, isolation, and cost. The agent image and interface are still identical; what's new is the machinery *around* it:

- A **gateway** in front that authenticates callers and only forwards `session_id`s they own — this is where the auth that `server.py` deliberately doesn't do finally happens.
- **Pod-per-session** routing (gateway → Redis → pod) so each conversation gets its own blast radius.
- A **standby pool** of pre-warmed pods so new sessions don't pay cold-start latency.
- **Egress lockdown** so the agent's `WebSearch` can only reach what you allow.

The full manifests and gateway live in [`hosting/kubernetes/`](./hosting/kubernetes/) — landed in a follow-up PR. The architecture walkthrough there explains why each piece exists.

---
## Making it production-ready

Two production concerns you can wire up in a few lines each — the notebook shows the code; the [hosting docs](https://code.claude.com/docs/en/agent-sdk/hosting) have the full production checklist (auth, graceful shutdown, idle-timeout tuning, autoscaling, cost controls).

### Observability

The SDK emits OpenTelemetry spans for every turn and tool call. Point it at your collector with two env vars — no code changes to `server.py` ([docs](https://code.claude.com/docs/en/agent-sdk/observability)):

In [ ]:
# In docker-compose.yml / modal_app.py / your k8s Deployment:
#   OTEL_EXPORTER_OTLP_ENDPOINT=http://your-collector:4317
#   OTEL_SERVICE_NAME=research-agent

### Liveness

`GET /health` is already in `server.py`. Point your orchestrator's liveness probe at it (compose `healthcheck:`, Modal health checks, k8s `livenessProbe`).

### Persistence beyond a volume

The `/data` volume mount is fine for single-host and Modal. For multi-host production, use a [`SessionStore` adapter](https://code.claude.com/docs/en/agent-sdk/session-storage) that mirrors transcripts to shared storage (S3, Postgres, Redis). Note: SessionStore is a *mirror* — the local disk write always happens first, and mirror failures emit `mirror_error` without interrupting the agent.

### Wire format

`server.py` streams raw SDK message types. That's fine for a cookbook; for a real API you'd define a stable wire format so SDK version bumps don't break clients.

---
## Choosing your tier

- **Tier 1** if it runs on one machine and you can restart it by hand.
- **Tier 2** if you want a URL and scale-to-zero without managing infra.
- **Tier 3** if you need multi-tenant isolation, network policy, or your compliance team says so.

For the full decision guide, see the [hosting docs](https://code.claude.com/docs/en/agent-sdk/hosting).

---
## Appendix — Porting to other providers

Same `hosting/Dockerfile`, different deploy command. Each of these exposes port 8000 and gives you a URL; mount something at `/data` for persistence.

**Fly Machines**
```bash
fly launch --dockerfile hosting/Dockerfile --no-deploy  # run from claude_agent_sdk/
fly volumes create data --size 1
fly deploy
```

**E2B**
```python
from e2b import Sandbox
sbx = Sandbox(template="research-agent")  # template built from hosting/Dockerfile
sbx.commands.run("./hosting/entrypoint.sh serve", background=True)
url = sbx.get_host(8000)
```

**Daytona**
```python
from daytona import Daytona, CreateSandboxFromImageParams
sbx = Daytona().create(CreateSandboxFromImageParams(image="research-agent"))
sbx.process.exec("./hosting/entrypoint.sh serve")
```

**Cloudflare Containers**
```ts
// wrangler.toml points at hosting/Dockerfile
export class Agent extends Container { defaultPort = 8000 }
```

**Vercel Sandbox**
```ts
import { Sandbox } from "@vercel/sandbox";
const sbx = await Sandbox.create({ image: "research-agent", ports: [8000] });
await sbx.runCommand({ cmd: "./hosting/entrypoint.sh", args: ["serve"], detached: true });
```